In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
import os
df_ideo= pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/df_mmxeu.csv')
def best_classifier_iterative_corrected(df, target_column, feature_range):
    """
    Evalúa modelos de clasificación (RF, XGBoost, LightGBM, CatBoost, SVM) 
    con diferentes números de características y selección de características por modelo.

    Args:
        df (pd.DataFrame): DataFrame con los datos.
        target_column (str): Nombre de la columna objetivo.
        feature_range (tuple): Rango de números de características a iterar (inicio, fin).
    """

    results = {}
    X = df.drop(target_column, axis=1)
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    classifiers = {
        "Random Forest": RandomForestClassifier(random_state=42),
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        "LightGBM": LGBMClassifier(random_state=42),
        "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
        "SVM": SVC(probability=True, random_state=42)
    }

    best_model = None
    best_auc = -float('inf')
    best_k = 0
    best_features = []
    best_classifier_name = ""

    for k in range(feature_range[0], feature_range[1] + 1):
        for classifier_name, classifier in classifiers.items():
            selector = SelectKBest(score_func=f_classif, k=k)
            X_train_selected = selector.fit_transform(X_train, y_train)
            X_test_selected = selector.transform(X_test)

            classifier.fit(X_train_selected, y_train)
            y_pred_prob = classifier.predict_proba(X_test_selected)[:, 1]
            auc = roc_auc_score(y_test, y_pred_prob)

            if auc > best_auc:
                best_auc = auc
                best_model = classifier
                best_k = k
                best_features = X.columns[selector.get_support()].tolist()
                best_classifier_name = classifier_name

    output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/App'
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, 'bestclassifier_iterative_corrected.txt'), 'w') as f:
        f.write(f"Mejor modelo: {best_classifier_name}\n")
        f.write(f"AUC: {best_auc}\n")
        f.write(f"Número de características: {best_k}\n")
        f.write(f"Características: {best_features}\n")

    return best_model, best_k, best_features, best_classifier_name

# Ejemplo de uso (asumiendo que df_ideo está definido):
best_model, best_k, best_features, best_classifier_name = best_classifier_iterative_corrected(df_ideo, 'ideologia', (10, 18))

print(f"Mejor modelo: {best_classifier_name}")
print(f"Número de características: {best_k}")
print(f"Características: {best_features}")

: 